# CF-Attn Architecture Sweep

Sweeps two hyperparameters of `CFAttnGaussianScoreNet`:
- **`cfattn_h`** — hidden / head dimension: 32, 64, 128, 256
- **`cfattn_K`** — number of spatial neighbors fed to attention: 4, 9, 16, 25

NeighborMLP, DSM, AMF, GMM-Levin are run at fixed defaults as anchors.
No Ridge. Each config runs on `SCENARIO_INDICES` scenarios (default 0 + 1).
Expected runtime on Colab T4: **~3–4 h** (16 configs × 2 scenarios × ~6 min).

In [ ]:
# ── Cell 1: Clone repo + install deps ─────────────────────────────────
import os, sys

REPO_URL      = 'https://github.com/michaelpiro/final-paper-experiment.git'
LOCAL_PROJECT = '/content/proj'
BRANCH        = 'iid-unified-experiment'

if os.path.exists(os.path.join(LOCAL_PROJECT, '.git')):
    !git -C {LOCAL_PROJECT} fetch --all -q
    !git -C {LOCAL_PROJECT} checkout {BRANCH} -q
    !git -C {LOCAL_PROJECT} pull origin {BRANCH} -q
else:
    !git clone -b {BRANCH} --depth 1 {REPO_URL} {LOCAL_PROJECT}

!pip install -q scikit-learn scipy tqdm matplotlib pyyaml pandas

for p in [LOCAL_PROJECT, os.path.join(LOCAL_PROJECT, 'experiments', 'spatial')]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(LOCAL_PROJECT)
print('Ready. CWD:', os.getcwd())
!git -C {LOCAL_PROJECT} log --oneline -1

In [ ]:
# ── Cell 2: GPU check ─────────────────────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — use Runtime > Change runtime type > GPU.')
print('PyTorch', torch.__version__, ' device =', DEVICE)

In [ ]:
# ── Cell 3: Mount Drive + link pavia-u.mat ────────────────────────────
RESULTS = '/content/results/cfattn_sweep'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS = '/content/drive/MyDrive/final_paper/cfattn_sweep'
    os.makedirs('/content/proj/data', exist_ok=True)
    DST = '/content/proj/data/pavia-u.mat'
    if not os.path.exists(DST):
        for src in ['/content/drive/MyDrive/final_paper/pavia-u.mat',
                    '/content/drive/MyDrive/final_paper/real_datasets/pavia-u.mat']:
            if os.path.exists(src):
                os.symlink(src, DST); print('Linked dataset from', src); break
        else:
            print('WARNING: pavia-u.mat not found on Drive — place it there.')
    else:
        print('Dataset already linked.')
except Exception as e:
    print('Drive not mounted:', repr(e))
os.makedirs(RESULTS, exist_ok=True)
assert os.path.exists('/content/proj/data/pavia-u.mat'), 'pavia-u.mat missing!'
print('RESULTS dir:', RESULTS)

In [ ]:
# ── Cell 4: Sweep config ──────────────────────────────────────────────
# Scenarios to average over.  0+1 = two manual boxes → stable estimates.
SCENARIO_INDICES = [0, 1]

# Base config — shared by every sweep config.
BASE = dict(
    # --- data ---
    dataset             = 'data/pavia-u.mat',
    norm_mode           = 'none',
    manual_boxes_path   = 'experiments/spatial/manual_boxes.json',
    amplitude           = 0.15,
    target_fraction     = 0.10,
    k                   = 5,
    # --- regularization ---
    local_scm_loading   = 1e-8,
    baseline_eig_floor  = 1e-12,
    whiten_eig_floor    = 1e-5,
    # --- CFAR ---
    cfar_bg_window      = 11,
    cfar_guard          = 1,
    cfar_lam            = 0.0,
    # --- CF-Attn defaults (will be overridden per sweep point) ---
    cfattn_h            = 64,
    cfattn_K            = 9,
    cfattn_epochs       = 1000,
    cfattn_lr           = 3e-4,
    cfattn_eps          = 1e-4,
    lam_ent             = 0.05,
    lam_div             = 0.05,
    lam_cov             = 1e-5,
    # --- NeighborMLP fixed at defaults (anchor) ---
    nmlp_d_lat          = 32,
    nmlp_K              = 8,
    nmlp_hidden         = 128,
    nmlp_n_layers       = 2,
    nmlp_epochs         = 1000,
    nmlp_lr             = 3e-4,
    nmlp_batch          = 256,
    # --- DSM ---
    dsm_hidden          = [128],
    dsm_epochs          = 1000,
    dsm_lr              = 5e-4,
    # --- Ridge: skip (1 epoch = essentially untrained, ignored in DET_ORDER) ---
    ridge_epochs        = 1,
    # --- GMM ---
    gmm_K               = 9,
    gmm_steps           = 50,
    # --- shared ---
    activation          = 'silu',
    dsm_sigma_rho       = 0.01,
    batch_size          = 256,
    weight_decay        = 1e-4,
    pfa_target          = 0.05,
    seed                = 42,
    random_scenario_seeds = [42, 123, 456, 789],
    min_pixels          = 4000,
)

# --- Sweep grid ---
# Each entry: (group_name, label, overrides_dict)
# We vary ONE param at a time; the default value is included so you can see
# where the default sits on the curve (marked with * in the analysis plots).

H_VALUES = [32, 64, 128, 256]   # cfattn_h
K_VALUES = [4, 9, 16, 25]       # cfattn_K

H_DEFAULT = 64
K_DEFAULT = 9

SWEEP = []
for h in H_VALUES:
    SWEEP.append(('h_sweep', f'h={h}', {'cfattn_h': h, 'cfattn_K': K_DEFAULT}))
for kk in K_VALUES:
    SWEEP.append(('K_sweep', f'K={kk}', {'cfattn_h': H_DEFAULT, 'cfattn_K': kk}))

print(f'Total configs: {len(SWEEP)}  × {len(SCENARIO_INDICES)} scenarios'
      f' = {len(SWEEP)*len(SCENARIO_INDICES)} runs')
print('Configs:')
for group, label, ov in SWEEP:
    print(f'  [{group}]  {label:12s}  overrides={ov}')

In [ ]:
# ── Cell 5: Runner helpers ────────────────────────────────────────────
import sys, os, glob, json, importlib, tempfile, time
import yaml
import numpy as np
import pandas as pd

# Detectors shown in this sweep (CF-Attn + CFAR, NeighborMLP as anchor,
# DSM + AMF + GMM-Levin as global anchors).  No Ridge.
SWEEP_DET_ORDER = [
    'DSM',
    'CF-Attn', 'CF-Attn-CFAR',
    'NeighborMLP', 'NeighborMLP-CFAR',
    'AMF',
    'GMM-Levin',
]

# Colors consistent with the main comparison notebook.
DET_COLORS = {
    'DSM':             '#ff7f0e',
    'CF-Attn':         '#aec7e8',
    'CF-Attn-CFAR':    '#1f77b4',
    'NeighborMLP':     '#2ca02c',
    'NeighborMLP-CFAR':'#006d2c',
    'AMF':             '#9467bd',
    'GMM-Levin':       '#e377c2',
}

def _reload_modules():
    for _m in ['dsm_model', 'cfattn_model', 'neighbor_mlp_model', 'local_detectors',
               'final_paper_experiments.models.neighbor_adapted',
               'run_colab', 'run_compare']:
        if _m in sys.modules:
            importlib.reload(sys.modules[_m])

def run_one(group, label, overrides, scenario_idx, results_root):
    """Train + evaluate one config on one scenario. Returns the run_dir path."""
    cfg = dict(BASE)
    cfg.update(overrides)
    cfg['scenario_index'] = int(scenario_idx)
    cfg['results_dir'] = results_root

    fd, tmp = tempfile.mkstemp(suffix='.yaml', prefix=f'cfattn_{label}_s{scenario_idx}_')
    os.close(fd)
    yaml.safe_dump(cfg, open(tmp, 'w'))

    argv = ['run_compare.py', '--config', tmp,
            '--results_dir', results_root,
            '--scenario', str(scenario_idx)]

    _reload_modules()
    import run_compare
    # Patch DET_ORDER so CF-Attn is active and Ridge is excluded.
    run_compare.DET_ORDER = SWEEP_DET_ORDER

    _argv = sys.argv
    sys.argv = argv
    t0 = time.time()
    try:
        run_compare.main()
    except SystemExit:
        pass
    finally:
        sys.argv = _argv

    elapsed = time.time() - t0
    # Find the run dir just created (latest compare_* in results_root).
    run_dir = sorted(glob.glob(os.path.join(results_root, 'compare_*')))[-1]
    # Tag it with group + label + scenario for easy lookup.
    tag_file = os.path.join(run_dir, 'sweep_tag.json')
    json.dump({'group': group, 'label': label, 'scenario': scenario_idx,
               'overrides': overrides, 'elapsed_s': round(elapsed)},
              open(tag_file, 'w'), indent=2)
    print(f'  → done in {elapsed/60:.1f} min  run_dir: {os.path.basename(run_dir)}')
    return run_dir

print('Helpers ready.')

In [ ]:
# ── Cell 6: RUN ALL (leave this running overnight) ────────────────────
# Iterates: scenario × sweep config.  Progress is printed live.
# Already-completed runs are NOT skipped automatically — comment out entries
# in SWEEP or SCENARIO_INDICES above if you want to resume partway.

run_log = []   # list of (group, label, scenario, run_dir)
n_total = len(SCENARIO_INDICES) * len(SWEEP)
n_done  = 0

for sidx in SCENARIO_INDICES:
    for group, label, overrides in SWEEP:
        n_done += 1
        print(f'\n[{n_done}/{n_total}]  scenario={sidx}  [{group}] {label}',
              flush=True)
        rdir = run_one(group, label, overrides, sidx, RESULTS)
        run_log.append((group, label, sidx, rdir))

print('\n=== ALL DONE ===')
for g, l, s, r in run_log:
    print(f'  scen={s}  [{g}] {l:12s}  {os.path.basename(r)}')

In [ ]:
# ── Cell 7: Load all results ──────────────────────────────────────────
# Scan RESULTS dir for every run that has a sweep_tag.json + metrics.json.

records = []
for run_dir in sorted(glob.glob(os.path.join(RESULTS, 'compare_*'))):
    tag_f    = os.path.join(run_dir, 'sweep_tag.json')
    metrics_f = os.path.join(run_dir, 'metrics.json')
    if not (os.path.exists(tag_f) and os.path.exists(metrics_f)):
        continue
    tag = json.load(open(tag_f))
    met = json.load(open(metrics_f))
    for row in met['rows']:
        records.append({
            'group':    tag['group'],
            'label':    tag['label'],
            'scenario': tag['scenario'],
            'cfattn_h': tag['overrides'].get('cfattn_h', H_DEFAULT),
            'cfattn_K': tag['overrides'].get('cfattn_K', K_DEFAULT),
            'Detector': row['Detector'],
            'pAUC@0.05': float(row['pAUC@0.05']),
            'AUC':       float(row['AUC']),
            'Pd@Pfa=0.05': float(row['Pd@Pfa=0.05']),
            'Pfa_avg':   float(row['Pfa_avg']),
            'run_dir':   run_dir,
        })

df_raw = pd.DataFrame(records)
# Average over scenarios
df = (df_raw
      .groupby(['group', 'label', 'cfattn_h', 'cfattn_K', 'Detector'])
      [['pAUC@0.05', 'AUC', 'Pd@Pfa=0.05', 'Pfa_avg']]
      .mean()
      .reset_index())

print(f'Loaded {len(df_raw)} records from {df_raw["run_dir"].nunique()} runs')
print(f'Groups: {df["group"].unique()}')
print(f'Detectors: {df["Detector"].unique().tolist()}')
df.head(12)

In [ ]:
# ── Cell 8a: Plot — AUC vs cfattn_h ──────────────────────────────────
import matplotlib.pyplot as plt

METRIC = 'AUC'      # swap to 'pAUC@0.05' or 'Pd@Pfa=0.05' as desired

sub = df[df['group'] == 'h_sweep'].sort_values('cfattn_h')
xs  = sorted(sub['cfattn_h'].unique())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for metric, ax in zip(['AUC', 'pAUC@0.05'], axes):
    for det in SWEEP_DET_ORDER:
        rows = sub[sub['Detector'] == det].sort_values('cfattn_h')
        if rows.empty:
            continue
        lw   = 2.0 if 'CF-Attn' in det else 1.2
        ls   = '--' if 'CFAR' in det else '-'
        ax.plot(rows['cfattn_h'], rows[metric],
                color=DET_COLORS.get(det, 'gray'), lw=lw, ls=ls,
                marker='o', markersize=5, label=det)
    ax.axvline(H_DEFAULT, color='k', lw=0.8, ls=':', alpha=0.5, label=f'default h={H_DEFAULT}')
    ax.set_xlabel('cfattn_h  (hidden / head dim)')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} vs cfattn_h  (K={K_DEFAULT} fixed)')
    ax.set_xticks(xs)
    ax.legend(fontsize=7, ncol=2)
    ax.grid(alpha=0.25)

fig.tight_layout()
out = os.path.join(RESULTS, 'sweep_h.png')
fig.savefig(out, dpi=150)
plt.show()
print('Saved:', out)

In [ ]:
# ── Cell 8b: Plot — AUC vs cfattn_K ──────────────────────────────────
sub = df[df['group'] == 'K_sweep'].sort_values('cfattn_K')
xs  = sorted(sub['cfattn_K'].unique())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for metric, ax in zip(['AUC', 'pAUC@0.05'], axes):
    for det in SWEEP_DET_ORDER:
        rows = sub[sub['Detector'] == det].sort_values('cfattn_K')
        if rows.empty:
            continue
        lw = 2.0 if 'CF-Attn' in det else 1.2
        ls = '--' if 'CFAR' in det else '-'
        ax.plot(rows['cfattn_K'], rows[metric],
                color=DET_COLORS.get(det, 'gray'), lw=lw, ls=ls,
                marker='o', markersize=5, label=det)
    ax.axvline(K_DEFAULT, color='k', lw=0.8, ls=':', alpha=0.5, label=f'default K={K_DEFAULT}')
    ax.set_xlabel('cfattn_K  (# spatial neighbors)')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} vs cfattn_K  (h={H_DEFAULT} fixed)')
    ax.set_xticks(xs)
    ax.legend(fontsize=7, ncol=2)
    ax.grid(alpha=0.25)

fig.tight_layout()
out = os.path.join(RESULTS, 'sweep_K.png')
fig.savefig(out, dpi=150)
plt.show()
print('Saved:', out)

In [ ]:
# ── Cell 8c: Plot — Pfa_avg vs h and K (false-alarm control) ─────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, group, xkey, xvals, xdefault, xlabel in [
    (axes[0], 'h_sweep', 'cfattn_h', H_VALUES, H_DEFAULT, 'cfattn_h'),
    (axes[1], 'K_sweep', 'cfattn_K', K_VALUES, K_DEFAULT, 'cfattn_K'),
]:
    sub = df[df['group'] == group].sort_values(xkey)
    for det in SWEEP_DET_ORDER:
        rows = sub[sub['Detector'] == det].sort_values(xkey)
        if rows.empty:
            continue
        lw = 2.0 if 'CF-Attn' in det else 1.2
        ls = '--' if 'CFAR' in det else '-'
        ax.plot(rows[xkey], rows['Pfa_avg'],
                color=DET_COLORS.get(det, 'gray'), lw=lw, ls=ls,
                marker='o', markersize=5, label=det)
    ax.axhline(0.05, color='k', lw=0.8, ls=':', alpha=0.5, label='target Pfa=0.05')
    ax.axvline(xdefault, color='k', lw=0.8, ls='--', alpha=0.3)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Pfa_avg')
    ax.set_title(f'Pfa_avg vs {xlabel}')
    ax.set_xticks(sorted(sub[xkey].unique()))
    ax.legend(fontsize=7, ncol=2)
    ax.grid(alpha=0.25)

fig.tight_layout()
out = os.path.join(RESULTS, 'sweep_pfa.png')
fig.savefig(out, dpi=150)
plt.show()
print('Saved:', out)

In [ ]:
# ── Cell 9: Summary table ─────────────────────────────────────────────
from IPython.display import display

pivot = df.pivot_table(
    index=['group', 'label', 'cfattn_h', 'cfattn_K'],
    columns='Detector',
    values='AUC'
).round(3)

# Reorder columns to match SWEEP_DET_ORDER
cols = [d for d in SWEEP_DET_ORDER if d in pivot.columns]
pivot = pivot[cols]

display(pivot.style
        .background_gradient(cmap='Greens', axis=None)
        .format('{:.3f}'))

out_csv = os.path.join(RESULTS, 'sweep_summary.csv')
pivot.to_csv(out_csv)
print('Saved:', out_csv)

# Also print the best h and best K for CF-Attn
for group, xkey in [('h_sweep', 'cfattn_h'), ('K_sweep', 'cfattn_K')]:
    sub = df[(df['group'] == group) & (df['Detector'] == 'CF-Attn-CFAR')]
    if not sub.empty:
        best = sub.loc[sub['AUC'].idxmax()]
        print(f'Best CF-Attn-CFAR AUC in {group}: {xkey}={int(best[xkey])}'
              f'  AUC={best["AUC"]:.3f}  pAUC={best["pAUC@0.05"]:.3f}')